# Graded Challenge 6 - Model Inference

**Nama:** Daffa Narendra Hutapea

**Batch:** HCK-040

**Objective:** Inference (separate from the training notebook, as required by the rubrics on graded challenge)

This notebook scores a new customer against the K-Means customer-segmentation model trained in `P1G6_daffa_hutapea.ipynb`.


## xi. Model Inference

### i. Import Libraries

In [1]:
import pandas as pd
import pickle

### ii. Load Trained Artifacts

We load the four artifacts that were serialized at the end of the training notebook:

| File | Purpose |
| --- | --- |
| `preprocessing_pipeline.pkl` | Fitted imputer + StandardScaler |
| `pca.pkl` | Fitted PCA (11 components, 95% variance) |
| `kmeans_model.pkl` | Final K-Means (k=3) |
| `feature_columns.pkl` | The exact column order the pipeline was trained on |


In [2]:
with open('preprocessing_pipeline.pkl', 'rb') as f:
    pipeline = pickle.load(f)

with open('pca.pkl', 'rb') as f:
    pca = pickle.load(f)

with open('kmeans_model.pkl', 'rb') as f:
    kmeans_model = pickle.load(f)

with open('feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

# Check the accuracy of the artifacts
print(f"Pipeline steps    : {list(pipeline.named_steps.keys())}")
print(f"PCA components    : {pca.n_components_}")
print(f"KMeans clusters   : k = {kmeans_model.n_clusters}")
print(f"Expected features : {feature_columns}")

Pipeline steps    : ['imputer', 'scaler']
PCA components    : 11
KMeans clusters   : k = 3
Expected features : ['BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE']


### iii. Define the New Customer

The data below is provided in the assignment README (the test customer with `CUST_ID = 9999`).

> **Note on `PURCHASES_FREQUENCY = 5`**:
> Every other `*_FREQUENCY` feature in this dataset (and in the assignment table) is bounded in `[0, 1]`. It represents a fraction of months in which a behavior occurred over a 6-12 month window. The value `5` provided for `PURCHASES_FREQUENCY` therefore sits well outside the variable's natural range.
>
> We treat this as the assignment intends, **using the value `5` as given without modification**, while flagging that this row will appear as a far-outlier in the scaled feature space and will likely pull the customer toward whichever cluster has the highest purchase activity. A real production pipeline would either clamp the value to `[0, 1]` or reject the row at validation time.


In [3]:
# New customer data from the README inference table
new_customer = {
    'CUST_ID': 9999,
    'BALANCE': 2000,
    'BALANCE_FREQUENCY': 0.5,
    'PURCHASES': 1000,
    'ONEOFF_PURCHASES': 600,
    'INSTALLMENTS_PURCHASES': 800,
    'CASH_ADVANCE': 500,
    'PURCHASES_FREQUENCY': 5,             # <-- anomalous (see note above); used as-given
    'ONEOFF_PURCHASES_FREQUENCY': 0.5,
    'PURCHASES_INSTALLMENTS_FREQUENCY': 0.7,
    'CASH_ADVANCE_FREQUENCY': 0.3,
    'CASH_ADVANCE_TRX': 3,
    'PURCHASES_TRX': 20,
    'CREDIT_LIMIT': 5000,
    'PAYMENTS': 800,
    'MINIMUM_PAYMENTS': 500,
    'PRC_FULL_PAYMENT': 0.2,
    'TENURE': 12,
}

# Build a 1-row DataFrame using the EXACT column order the pipeline was trained on.
# We drop CUST_ID because it was excluded from training features.
new_df = pd.DataFrame([new_customer])[feature_columns]
new_df


,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,2000,0.5,1000,600,800,500,5,0.5,0.7,0.3,3,20,5000,800,500,0.2,12


### iv. Run the Inference Pipeline

Three steps, mirroring the training pipeline:

1. **Preprocessing** — imputer + StandardScaler (using the training-set statistics)
2. **PCA projection** — into the 11-component subspace
3. **K-Means prediction** — assign the nearest cluster centroid


In [4]:
# Step 1: preprocess (impute + scale)
new_scaled = pipeline.transform(new_df)

# Step 2: PCA projection
new_pca = pca.transform(new_scaled)

# Step 3: cluster prediction
predicted_cluster = int(kmeans_model.predict(new_pca)[0])

# Distance to each centroid (for transparency)
distances = kmeans_model.transform(new_pca)[0]

print(f"Predicted cluster : {predicted_cluster}")
print()
print("Distance to each centroid (closer = more similar):")
for c, d in enumerate(distances):
    marker = '  <-- predicted' if c == predicted_cluster else ''
    print(f"  Cluster {c}: {d:.3f}{marker}")


Predicted cluster : 1

Distance to each centroid (closer = more similar):
  Cluster 0: 10.493
  Cluster 1: 8.067  <-- predicted
  Cluster 2: 9.958


### v. Persona Interpretation

The customer profile is read off against the three personas defined during model evaluation:

| Cluster | Persona | Behavior |
| ---: | --- | --- |
| **0** | **Revolvers / Cash-Advance Heavy** | High balance + high cash-advance + very low full-payment ratio. Treat as borrowing-driven; primary risk segment. |
| **1** | **Active Transactors** | High purchases + high purchase frequency + healthier repayment. Premium-tier candidate; highest growth potential. |
| **2** | **Dormant / Low-Engagement** | Low balance + low purchases + low payments overall; pays in full when used. Reactivation candidate. |


In [5]:
personas = {
    0: ("Revolvers / Cash-Advance Heavy",
        "Customer carries a high balance and uses the card primarily for cash advances. "
        "Low full-payment ratio. Recommended action: balance-transfer or lower-APR loan offers; "
        "monitor for credit risk."),
    1: ("Active Transactors",
        "Customer makes frequent and varied purchases (oneoff + installments) with a healthier "
        "repayment pattern. Recommended action: reward-tier upgrade, premium-card upsell, "
        "partner-merchant promotions."),
    2: ("Dormant / Low-Engagement",
        "Customer barely uses the card. When they do use it, they tend to pay in full. "
        "Recommended action: reactivation campaign (welcome-back bonus, fee waivers, "
        "first-N-transaction cashback)."),
}

persona_name, persona_desc = personas[predicted_cluster]

print(f"=== Inference Result for CUST_ID = {new_customer['CUST_ID']} ===")
print(f"Predicted Cluster : {predicted_cluster}")
print(f"Persona           : {persona_name}")


=== Inference Result for CUST_ID = 9999 ===
Predicted Cluster : 1
Persona           : Active Transactors


### Discovery:

Customer is selected to be at cluster 1, which is an 'Active Transactors'. This customer makes frequent and varied purchases (oneoff + installments) with a healthier repayment pattern. 

**Recommended action**: reward-tier upgrade, premium-card upsell, partner-merchant promotions.

### vi. Sanity Check vs the Test Customer's Raw Profile

Manual reasoning against the persona table:

- This customer has **moderate balance ($2,000)** and **moderate cash advance ($500)**, not extreme on either axis.
- They have **high purchase activity** (`PURCHASES` = $1,000, `PURCHASES_TRX` = 20) with a mix of one-off and installment buying.
- The reported `PURCHASES_FREQUENCY = 5` is an outlier value (see the note in section iii) that will scale into a very large standardized value, pushing the row strongly toward the high-purchase-frequency cluster (Cluster 1: Active Transactors).
- `PRC_FULL_PAYMENT` = 0.2 is consistent with Cluster 1's repayment profile.

A predicted cluster of **1 (Active Transactors)** is the expected outcome given the customer's spending profile, and the anomalous `PURCHASES_FREQUENCY` only reinforces the assignment.

> If the assignment was meant to be `PURCHASES_FREQUENCY = 0.5` (a typo), the same logic applies. The customer would still most likely land in Cluster 1 because of the high transaction count and balanced purchase types, just with less geometric extremity in the scaled space.
